# Evaluation — Multi-Agent Recruiting Chatbot

Measures how well the system predicts the correct action (`continue` / `schedule` / `end`)
on the labeled dataset `data/sms_conversations.json`.

Follows the course evaluation pattern from `Course17/ML - Evaluation & Interpretation.ipynb`
and `Course23/Nlp.ipynb`.

**The task.** Given a conversation history up to and including a candidate turn, predict the
label of the **next recruiter turn**. Only recruiter turns are labeled; candidate turns are
always `null`.

**Four things this notebook must always report** (CLAUDE.md section 10):

1. Accuracy **and the majority-class baseline next to it** — an accuracy number without that
   reference point is unreadable.
2. The confusion matrix, with real class names on both axes.
3. Per-class metrics — `end` is the smallest class at 15 examples, and overall accuracy can
   hide a total failure on it.
4. The misclassified turns, with history, true label and prediction.

**Split rule.** Splitting happens at the **conversation** level, never the turn level. Turns
inside one conversation share a history prefix, so a turn-level split leaks the test set into
both training and few-shot examples. The same split and seed are used by
`scripts/run_finetuning.py`, so the fine-tuned Exit Advisor never sees the held-out data.

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from app.config import settings
from app.modules.main_agent.actions import ACTIONS
from app.modules.evaluation.dataset import (
    MAJORITY_BASELINE,
    build_decision_points,
    load_conversations,
    split_by_conversation,
)
from app.modules.evaluation.metrics import evaluate, error_table, plot_confusion, predict_all

pd.set_option("display.max_colwidth", 120)
print(settings)

## 1. Load the dataset

15 conversations, 103 turns, 59 of them labeled.

In [ ]:
conversations = load_conversations()
points = build_decision_points(conversations)

print(f"Conversations : {len(conversations)}")
print(f"Turns         : {sum(len(c['turns']) for c in conversations)}")
print(f"Decision points: {len(points)}")

In [ ]:
# Label distribution. `continue` is the majority class at ~42.4%.
labels = pd.Series([p.label for p in points], name="label")
distribution = labels.value_counts().reindex(list(ACTIONS))
display(distribution.to_frame("count").assign(share=lambda d: (d["count"] / d["count"].sum()).round(3)))

print(f"Majority-class baseline: {MAJORITY_BASELINE:.3f}")

### Label semantics — read before interpreting anything below

| Label | Fires when |
|---|---|
| `continue` | Information is being exchanged — asking about experience, answering a question about the role. |
| `schedule` | A slot is being **proposed or renegotiated**. |
| `end` | The conversation is **over**, either way. |

`end` is **terminal, not negative**. It covers 11 happy endings ("your interview is confirmed")
and 4 opt-outs ("I'll close your application"). A model that equates `end` with "the candidate
is uninterested" gets most of the class wrong — expect that to show up as `end` → `schedule`
confusion in the matrix below.

## 2. Split by conversation

Held-out conversations are never used for training the Exit Advisor and never appear in a
few-shot example. Stratified by ending flavour so both opt-outs and bookings land on each side.

In [ ]:
train_ids, test_ids = split_by_conversation(conversations, test_size=5, seed=42)

test_points = [p for p in points if p.conversation_id in test_ids]
train_points = [p for p in points if p.conversation_id in train_ids]

print(f"Train conversations: {sorted(train_ids)}  ->  {len(train_points)} decision points")
print(f"Test  conversations: {sorted(test_ids)}  ->  {len(test_points)} decision points")

assert not (set(train_ids) & set(test_ids)), "Splits overlap — the evaluation would be invalid."

## 3. Run the system

Predictions are cached to disk, so re-running this notebook does not re-spend tokens.
Delete the cache file to force a fresh run.

In [ ]:
CACHE = settings.data_dir / "cache" / "eval_predictions.json"
CACHE.parent.mkdir(parents=True, exist_ok=True)

predictions = predict_all(test_points, cache_path=CACHE)
print(f"Predicted {len(predictions)} turns.")

## 4. Metrics

In [ ]:
result = evaluate(test_points, predictions)

print(f"Accuracy            : {result.accuracy:.3f}")
print(f"Majority baseline   : {result.baseline_accuracy:.3f}")
print(f"Lift over baseline  : {result.accuracy - result.baseline_accuracy:+.3f}")

### Confusion matrix

Rows are the true label, columns the prediction. The cell to watch is `end` → `schedule`:
that is the model failing to recognise a confirmed booking as terminal.

In [ ]:
display(result.confusion)
plot_confusion(result, title="Action prediction — test split")
plt.show()

### Per-class metrics

`end` is the smallest class. Check its recall specifically — overall accuracy can look healthy
while `end` is being missed entirely.

In [ ]:
display(result.report)

## 5. Error analysis

The error table is worth more than the headline score: a systematic confusion between two
classes points at a prompt fix, while scattered errors point at genuine ambiguity.

In [ ]:
errors = error_table(test_points, predictions)
print(f"{len(errors)} misclassified out of {len(test_points)}")
display(errors)

In [ ]:
# Which confusions dominate?
if len(errors):
    display(
        errors.groupby(["true_label", "predicted_label"]).size()
        .sort_values(ascending=False).to_frame("count")
    )

## 6. Conclusions

_Fill in after the first real run:_

- Accuracy vs baseline — is the lift meaningful given only ~20 test decision points?
- Which class is weakest, and does the error table suggest a prompt fix or genuine ambiguity?
- Did the fine-tuned Exit Advisor beat the few-shot fallback? Re-run with
  `FT_EXIT_ADVISOR_MODEL` unset to get the comparison.
- Sample-size caveat: 15 conversations is small. Treat differences of one or two turns as noise.